# Panel de empresas por trimestre

El notebook arma el panel: una fila por empresa y trimestre calendario, una columna por concepto y por ratio. Sigue `docs/rumbo-mai.pdf`:

- Período 2018Q1–2024Q4.
- Empresas que quiebran y que no, pero no las que se crearon durante el período.
- Panel no balanceado: una empresa no necesita datos en todos los trimestres.
- Nombres de columnas en español.

Pasos:

1. Leer los documentos y las empresas.
2. Quedarse con los tags de `shared/concepts.py`.
3. Calcular los flujos trimestrales.
4. Asignar cada valor a su trimestre calendario.
5. Elegir un tag por empresa, concepto y trimestre.
6. Construir la ganancia bruta cuando falta.
7. Quedarse con las empresas que existían al inicio.
8. Cortar a las empresas que quiebran.
9. Calcular los ratios.
10. Guardar `data/panel.csv`.

Entradas: `data/sub.csv` y `data/num.csv` (salen de `build_dataset.ipynb`) y `data/bankruptcy.csv` (sale de `bankruptcy.ipynb`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


if "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")

    CANDIDATOS = [
        Path("/content/drive/MyDrive/Colab Notebooks/Diplodatos MAI Grupo 6"),
        Path("/content/drive/MyDrive/Diplodatos MAI Grupo 6"),
    ]
    WORKING_DIR = next((p for p in CANDIDATOS if p.exists()), None)
    if WORKING_DIR is None:
        raise FileNotFoundError(
            "No encuentro la carpeta del grupo. ¿Agregaste el acceso directo "
            "en 'Mi unidad' desde 'Compartido conmigo'?"
        )
    sys.path.append(str(WORKING_DIR))  # to import `shared`
else:
    WORKING_DIR = Path.cwd()

from shared.concepts import CONCEPTS, FLOWS

## Paso 1: documentos y empresas

`quarter()` asigna una fecha de cierre a su trimestre calendario con la regla de `docs/decisions.md`, sección 11: el trimestre que contiene más meses del trimestre fiscal. En código: restar un mes y tomar el trimestre. Por ejemplo, un cierre el 31 de enero de 2019 va a 2018Q4.

Cada empresa tiene un solo sector y un solo nombre: los de su documento más reciente por `filed` (`docs/decisions.md`, sección 2).

In [2]:
START, END = pd.Period("2018Q1"), pd.Period("2024Q4")


def quarter(month: pd.Series) -> pd.Series:
    """Calendar quarter with most months of the fiscal quarter that ends in `month` (decisions.md, section 11)."""
    return (month - 1).dt.asfreq("Q")


sub = pd.read_csv(f"{WORKING_DIR}/data/sub.csv", dtype=str)
sub["month"] = pd.to_datetime(sub["period"], format="%Y%m%d").dt.to_period("M")

# name and sector of the most recent document (`filed` is AAAAMMDD, so text order = date order)
companies = sub.sort_values("filed").groupby("cik")[["name", "sector"]].last()
companies["sector"] = companies["sector"].map({"health": "salud", "energy": "energia"})
companies["sector"].value_counts()

sector
energia    327
salud      184
Name: count, dtype: int64

## Paso 2: tags de `concepts.py`

`CONCEPTS` asigna a cada concepto una lista de tags. El orden de cada lista es la prioridad: primero el tag principal, después los alternativos (`docs/decisions.md`, sección 12).

- **Flujos** (`FLOWS`): miden un período, por ejemplo los ingresos de un trimestre. Tienen `qtrs` de 1 a 4.
- **Saldos**: miden una fecha, por ejemplo los activos al cierre. Tienen `qtrs` = 0.

La celda lee de `num.csv` solo las filas con un tag de `CONCEPTS`. `filed` es la fecha en que se conoció cada número (`docs/decisions.md`, sección 8).

In [3]:
# tag -> concept, and tag -> position in its list (0 = principal tag)
CONCEPT = {t: c for c, tags in CONCEPTS.items() for t in tags}
RANK = {t: i for tags in CONCEPTS.values() for i, t in enumerate(tags)}

num = pd.read_csv(
    f"{WORKING_DIR}/data/num.csv",
    dtype={"cik": str, "ddate": str, "filed": str},
    usecols=["cik", "tag", "ddate", "qtrs", "value", "filed"],
)
num = num[num["tag"].isin(CONCEPT)]
num["concept"] = num["tag"].map(CONCEPT)
num["month"] = pd.to_datetime(num["ddate"], format="%Y%m%d").dt.to_period("M")
num.groupby("qtrs").size()

qtrs
0    60785
1    33952
2    12170
3    12061
4    13289
dtype: int64

## Paso 3: flujos trimestrales

El panel usa el flujo de un trimestre (`docs/decisions.md`, sección 10). Muchos documentos no lo traen: traen el acumulado desde el inicio del año fiscal.

- Si hay valor con `qtrs` = 1, se usa ese valor.
- Si no, se resta: acumulado de k trimestres menos acumulado de k − 1 trimestres que termina 3 meses antes. Los dos acumulados empiezan en el mismo mes.

Ejemplos:

- **Cuarto trimestre:** año (`qtrs` = 4, diciembre) menos 9 meses (`qtrs` = 3, septiembre). El 10-K no trae el cuarto trimestre solo.
- **Flujo operativo del segundo trimestre:** 6 meses (`qtrs` = 2, junio) menos 3 meses (`qtrs` = 1, marzo). Los 10-Q suelen traer este flujo solo acumulado.

La resta se hace con el mismo tag (`docs/decisions.md`, sección 12, regla A). Nunca mezcla dos tags. El valor restado conserva el `filed` del acumulado de k trimestres, que es el más nuevo.

La celda mueve cada acumulado de k − 1 trimestres 3 meses hacia adelante y lo une a la fila del acumulado de k trimestres. Después resta. La tabla cuenta los valores trimestrales según su origen: `qtrs` = 1 es un valor reportado; 2, 3 y 4 son restas.

In [4]:
flows = num[num["concept"].isin(FLOWS)]

# the accumulated k-1 quarters, moved to the row of the accumulated k quarters (3 months later)
previous = flows.assign(month=flows["month"] + 3, qtrs=flows["qtrs"] + 1)
flows = flows.merge(
    previous[["cik", "tag", "month", "qtrs", "value"]],
    on=["cik", "tag", "month", "qtrs"],
    how="left",
    suffixes=("", "_previous"),
)
flows["value"] = flows["value"].where(flows["qtrs"] == 1, flows["value"] - flows["value_previous"])

# ponytail: "3 months before" compares months, so a 52/53-week year ending on the 1st of a month
# can miss its previous accumulated value; compare days if that matters
flows = flows.dropna(subset="value")
flows.groupby("qtrs").size()

qtrs
1    33952
2    11614
3    11495
4    10047
dtype: int64

## Verificación de la resta

Algunas fechas tienen el trimestre reportado (`qtrs` = 1) y también la resta de acumulados. La celda compara los dos valores. Si coinciden, la resta es correcta.

- **1 USD:** el valor es idéntico.
- **1.000 USD:** muchas empresas redondean a miles, así que la resta puede diferir en el último dígito.
- **1%:** una diferencia mayor indica otro problema, por ejemplo versiones de documentos distintos (`docs/decisions.md`, sección 8).

In [5]:
reported = flows[flows["qtrs"] == 1]
subtracted = flows[flows["qtrs"] > 1]
both = reported.merge(subtracted, on=["cik", "tag", "month"], suffixes=("_reported", "_subtracted"))
difference = (both["value_reported"] - both["value_subtracted"]).abs()
print(f"fechas con los dos valores: {len(both)}")
print(f"diferencia <= 1 USD: {(difference <= 1).mean():.1%}")
print(f"diferencia <= 1.000 USD: {(difference <= 1000).mean():.1%}")
print(f"diferencia <= 1%: {(difference <= 0.01 * both['value_reported'].abs()).mean():.1%}")

fechas con los dos valores: 18749
diferencia <= 1 USD: 90.6%
diferencia <= 1.000 USD: 94.1%
diferencia <= 1%: 95.7%


## Paso 4: trimestre calendario

Cada valor va al trimestre de su fecha (`quarter()`, paso 1). Antes se eliminan las fechas que no son el cierre de un documento de la empresa (`sub.period`, `docs/decisions.md`, sección 11). Dos casos:

- **Comparativos con otro calendario.** Sustainable Projects cambió su cierre de mayo a diciembre en 2019. Sus documentos nuevos traen comparativos con cierre en diciembre de 2018, y el trimestre 2018Q4 ya tiene el cierre de noviembre.
- **Subperíodos por quiebra.** Una empresa que sale de una quiebra (por ejemplo, Gulfport Energy en 2021) reporta el período antes y después de la salida por separado.

Una fecha puede tener el trimestre reportado y una o más restas. Se usa el reportado; si no hay, la resta con menos trimestres acumulados.

Una empresa que cambia el cierre de su año fiscal puede tener dos documentos con cierre en el mismo trimestre. En ese caso se usa el cierre más reciente. La celda cuenta cuántos valores se descartan por esa razón.

In [6]:
# a date can have the reported quarter and one or more subtractions: keep the smallest `qtrs` (reported first)
flows = flows.sort_values("qtrs").drop_duplicates(["cik", "tag", "month"])
stocks = num[~num["concept"].isin(FLOWS) & (num["qtrs"] == 0)]
values = pd.concat([flows, stocks])[["cik", "tag", "concept", "month", "value", "filed"]]

# keep only the closing dates of the company's documents
values = values.merge(sub[["cik", "month"]].drop_duplicates(), on=["cik", "month"])
values["quarter"] = quarter(values["month"])
values = values[values["quarter"].between(START, END)]

# two closing dates in the same quarter (change of fiscal year): keep the latest
before = len(values)
values = values.sort_values("month").drop_duplicates(["cik", "tag", "quarter"], keep="last")
print(f"valores descartados por dos cierres en el mismo trimestre: {before - len(values)}")
values

valores descartados por dos cierres en el mismo trimestre: 10


,cik,tag,concept,month,value,filed,quarter
263,862692,NetCashProvidedByUsedInOperatingActivitiesCont...,flujo_caja_operativo,2018-02,1691863.0,20180416,2018Q1
269,862692,Revenues,ingresos_netos,2018-02,6228118.0,20180416,2018Q1
316,1446152,OperatingIncomeLoss,resultado_operativo,2018-02,-187295.0,20180423,2018Q1
23270,1495648,NetIncomeLoss,resultado_neto,2018-02,-41145.0,20220712,2018Q1
47971,1603793,AssetsCurrent,activos_corrientes,2018-02,353641.0,20190613,2018Q1
...,...,...,...,...,...,...,...
89708,1898766,Liabilities,pasivos_totales,2025-01,1519749.0,20250314,2024Q4
89370,1978111,AssetsCurrent,activos_corrientes,2025-01,27474.0,20250227,2024Q4
89368,1978111,LiabilitiesCurrent,pasivos_corrientes,2025-01,62503.0,20250227,2024Q4
89367,1978111,CashCashEquivalentsRestrictedCashAndRestricted...,caja,2025-01,27474.0,20250227,2024Q4


## Paso 5: un tag por empresa, concepto y trimestre

En cada empresa, concepto y trimestre se usa el primer tag de `CONCEPTS` con valor (regla A, `docs/decisions.md`, sección 12). Una empresa puede cambiar de tag entre trimestres.

`panel` tiene una fila por empresa y trimestre, y una columna por concepto. NaN significa que ningún tag del concepto tiene valor.

`fecha_presentacion` es el `filed` más reciente de los valores de la fila: la fecha desde la que se conoce la fila completa.

In [7]:
# principal tag first: sort by position in the list, keep the first row of each concept and quarter
chosen = values.sort_values("tag", key=lambda t: t.map(RANK)).drop_duplicates(["cik", "concept", "quarter"])
panel = chosen.pivot(index=["cik", "quarter"], columns="concept", values="value")[list(CONCEPTS)]
panel["fecha_presentacion"] = chosen.groupby(["cik", "quarter"])["filed"].max()
panel

concept          ingresos_netos  resultado_operativo  resultado_neto  \
cik     quarter                                                        
1000229 2018Q1      170018000.0           32316000.0      23527000.0   
        2018Q2      175475000.0           33449000.0      24752000.0   
        2018Q3      182146000.0           34879000.0      22412000.0   
        2018Q4      173207000.0           17978000.0       8835000.0   
        2019Q1      169194000.0           16414000.0      40510000.0   
...                         ...                  ...             ...   
96793   2023Q4        8510000.0            -433000.0      -3075000.0   
        2024Q1        7462000.0            -853000.0      -1396000.0   
        2024Q2        7913000.0            -675000.0       4288000.0   
        2024Q3        7923000.0           -1194000.0       -549000.0   
        2024Q4        7935000.0           -1012000.0      -1343000.0   

concept          activos_totales  activos_corrientes  pasivos_totales  \
cik     quarter                                                         
1000229 2018Q1       597140000.0         220185000.0              NaN   
        2018Q2       604744000.0         225662000.0              NaN   
        2018Q3       666597000.0         239125000.0              NaN   
        2018Q4       648827000.0         230977000.0              NaN   
        2019Q1       791760000.0         236423000.0              NaN   
...                          ...                 ...              ...   
96793   2023Q4        18787000.0          13710000.0              NaN   
        2024Q1        16144000.0          10966000.0              NaN   
        2024Q2        20612000.0          15664000.0              NaN   
        2024Q3        19629000.0          15616000.0              NaN   
        2024Q4        17872000.0          14206000.0              NaN   

concept          pasivos_corrientes  patrimonio_neto  flujo_caja_operativo  \
cik     quarter                                                              
1000229 2018Q1          104327000.0      147638000.0            23093000.0   
        2018Q2          100591000.0      153783000.0            26990000.0   
        2018Q3          103509000.0      157438000.0            23843000.0   
        2018Q4          102547000.0      156759000.0            37901000.0   
        2019Q1          128615000.0      183165000.0            25156000.0   
...                             ...              ...                   ...   
96793   2023Q4            5228000.0              NaN             -393000.0   
        2024Q1            3959000.0              NaN            -1062000.0   
        2024Q2            4213000.0              NaN             -386000.0   
        2024Q3            3851000.0              NaN             -915000.0   
        2024Q4            3495000.0              NaN            -1167000.0   

concept          ganancia_bruta        caja  costo_ventas fecha_presentacion  
cik     quarter                                                               
1000229 2018Q1              NaN  13244000.0           NaN           20200210  
        2018Q2              NaN  13021000.0           NaN           20200210  
        2018Q3              NaN  14100000.0           NaN           20200210  
        2018Q4              NaN  13116000.0           NaN           20210208  
        2019Q1              NaN  13206000.0           NaN           20210208  
...                         ...         ...           ...                ...  
96793   2023Q4              NaN   2055000.0           NaN           20240213  
        2024Q1              NaN   1211000.0           NaN           20240515  
        2024Q2              NaN   7170000.0           NaN           20240930  
        2024Q3              NaN   7529000.0           NaN           20241112  
        2024Q4              NaN   8020000.0           NaN           20250212  

[7690 rows x 13 columns]

## Paso 6: ganancia bruta

Si falta `GrossProfit`, ganancia bruta = ingresos netos − costo de ventas, como indica la guía de la mentora (`docs/decisions.md`, sección 12). El costo de ventas solo sirve para esto y no queda en el panel.

Después se eliminan las filas que quedaron sin ningún concepto (las que solo tenían costo de ventas).

In [8]:
constructed = panel["ganancia_bruta"].isna() & (panel["ingresos_netos"] - panel["costo_ventas"]).notna()
panel["ganancia_bruta"] = panel["ganancia_bruta"].fillna(panel["ingresos_netos"] - panel["costo_ventas"])
panel = panel.drop(columns="costo_ventas")

CONCEPT_COLUMNS = [c for c in CONCEPTS if c != "costo_ventas"]
panel = panel.dropna(subset=CONCEPT_COLUMNS, how="all")
print(f"ganancia bruta reportada: {panel['ganancia_bruta'].notna().sum() - constructed.sum()}")
print(f"ganancia bruta construida: {constructed.sum()}")

ganancia bruta reportada: 1188
ganancia bruta construida: 340


## Paso 7: empresas que existían al inicio

El rumbo deja afuera a las empresas que se crearon durante el período. Entra una empresa si tiene al menos un concepto con valor en 2018Q1 (`docs/decisions.md`, sección 13).

Como el paso 4 solo conserva los cierres de documentos de la empresa, en la práctica es una empresa con un 10-K o 10-Q con cierre en 2018Q1.

In [9]:
existing = panel.xs(START, level="quarter").index
panel = panel[panel.index.get_level_values("cik").isin(existing)]
companies.loc[existing, "sector"].value_counts()

sector
energia    246
salud       86
Name: count, dtype: int64

## Paso 8: quiebras

La fecha de quiebra es la del 8-K con el ítem 1.03, de `bankruptcy.ipynb` (`data/bankruptcy.csv`). En el panel, una empresa que quiebra deja de tener filas desde el trimestre calendario de la quiebra (`docs/decisions.md`, sección 13). Por ejemplo, una quiebra del 2021-08-15 deja la última fila en 2021Q2.

El trimestre de la quiebra es el que contiene la fecha. No se resta un mes, porque es una fecha de un evento y no el cierre de un trimestre fiscal.

`fecha_quiebra` llega hasta el 31/12/2025. Una quiebra de 2025 no corta el panel, pero sirve para etiquetar la quiebra en los próximos 2 trimestres o 1 año.

La tabla cuenta, por sector, las empresas con quiebra dentro del panel (2018–2024) y en 2025.

In [10]:
bankruptcy = pd.read_csv(f"{WORKING_DIR}/data/bankruptcy.csv", dtype=str).set_index("cik")["bankruptcy_date"]
companies["fecha_quiebra"] = bankruptcy

bankruptcy_quarter = pd.to_datetime(companies["fecha_quiebra"]).dt.to_period("Q")
cik, quarters = panel.index.get_level_values("cik"), panel.index.get_level_values("quarter")
cut = quarters >= bankruptcy_quarter.reindex(cik).to_numpy()  # NaT compares False: no bankruptcy, no cut
print(f"filas eliminadas por quiebra: {cut.sum()}")
panel = panel[~cut]

in_panel = companies.loc[existing]
in_panel.groupby("sector")["fecha_quiebra"].agg(
    quiebra_2018_2024=lambda d: (d <= "2024-12-31").sum(),
    quiebra_2025=lambda d: (d > "2024-12-31").sum(),
)

filas eliminadas por quiebra: 254


,quiebra_2018_2024,quiebra_2025
sector,,
energia,46,0
salud,8,0


## Paso 9: ratios

Los ratios van en fracción, no en porcentaje (`docs/rumbo-mai.pdf`). El ROA y el ROE usan el resultado del trimestre, sin anualizar (`docs/decisions.md`, sección 10).

Una división por cero (por ejemplo, ingresos 0 en una empresa sin ventas) da infinito. Se reemplaza por NaN.

In [11]:
RATIOS = {
    "roe": ("resultado_neto", "patrimonio_neto"),
    "roa": ("resultado_neto", "activos_totales"),
    "endeudamiento": ("pasivos_totales", "activos_totales"),
    "liquidez_corriente": ("activos_corrientes", "pasivos_corrientes"),
    "fco_ingresos": ("flujo_caja_operativo", "ingresos_netos"),
    "margen_bruto": ("ganancia_bruta", "ingresos_netos"),
    "margen_operativo": ("resultado_operativo", "ingresos_netos"),
    "margen_neto": ("resultado_neto", "ingresos_netos"),
}
for ratio, (numerator, denominator) in RATIOS.items():
    panel[ratio] = (panel[numerator] / panel[denominator]).replace([np.inf, -np.inf], np.nan)

panel[list(RATIOS)].describe().T

,count,mean,std,min,25%,50%,75%,max
concept,,,,,,,,
roe,5188.0,0.124489,6.353678,-71.893672,-0.047757,0.015605,0.067595,342.498509
roa,5665.0,-2.657234,196.358869,-7688.500000,-0.050911,-0.002444,0.017499,11483.544118
endeudamiento,4412.0,78.733379,1935.879492,0.001930,0.281001,0.490738,0.861560,103962.500000
liquidez_corriente,5749.0,3.213558,9.245344,0.000000,0.618833,1.248983,2.427929,359.403049
fco_ingresos,4592.0,-4.128456,118.407099,-5406.156250,-0.008284,0.173572,0.497201,140.167472
margen_bruto,846.0,0.018489,6.207728,-178.000000,0.140390,0.301322,0.517045,8.853026
margen_operativo,3904.0,-39.015432,2028.143730,-126416.333333,-0.366329,0.031413,0.216684,45.358579
margen_neto,4564.0,-52.827579,2348.352409,-143664.333333,-0.388771,0.003508,0.145152,389.842233


## Paso 10: guardar

`data/panel.csv` tiene una fila por empresa y trimestre con datos: `cik`, `nombre`, `sector`, `trimestre`, `fecha_presentacion`, los 11 conceptos (en USD), los 8 ratios y `fecha_quiebra` (vacía si la empresa no quiebra hasta 2025).

El panel no es balanceado: una empresa no tiene filas antes de su primer documento con datos, después de su último documento o desde su quiebra, y puede tener trimestres sueltos sin datos.

La tabla cuenta las empresas y las filas por sector, y el porcentaje de filas con cada concepto.

In [12]:
out = panel.reset_index().merge(companies, left_on="cik", right_index=True)
out = out.rename(columns={"name": "nombre", "quarter": "trimestre"})
out = out[["cik", "nombre", "sector", "trimestre", "fecha_presentacion", *CONCEPT_COLUMNS, *RATIOS, "fecha_quiebra"]]

assert not out.duplicated(["cik", "trimestre"]).any()
assert out["trimestre"].between(START, END).all()
out.to_csv(f"{WORKING_DIR}/data/panel.csv", index=False)

print(out.groupby("sector").agg(empresas=("cik", "nunique"), filas=("cik", "size")))
out.groupby("sector")[CONCEPT_COLUMNS].apply(lambda d: d.notna().mean().round(2)).T

         empresas  filas
sector                  
energia       245   4129
salud          86   1686


sector,energia,salud
ingresos_netos,0.84,0.79
resultado_operativo,0.83,0.89
resultado_neto,0.98,0.99
activos_totales,1.00,1.00
activos_corrientes,0.99,0.99
pasivos_totales,0.75,0.81
pasivos_corrientes,1.00,0.99
patrimonio_neto,0.88,0.98
flujo_caja_operativo,1.00,0.99
ganancia_bruta,0.11,0.34
